# AF2CUE1 — paired source-parent bootstrap
Validation-only inference atas checkpoint tetap AF2BASE, AF2SPDS, dan AF2CUE1. Seluruh sibling image dari parent yang sama di-bootstrap sebagai satu cluster. Tidak ada training dan test tidak tersedia.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys, tarfile, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU untuk tiga inference validation.'
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/af2-radially-normalized-angular-density'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],cwd=WORK,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
ARCHIVE_REL='bundles/faruq-development-v3-grouped.tar'
BASE_REL='experiments/faruq-v3-af2-spds-v1/val_reports/AF2BASE_seed42_result.json'
SPDS_REL='experiments/faruq-v3-af2-spds-v1/val_reports/AF2SPDS_seed42_result.json'
CUE_REL='experiments/faruq-v3-af2-spds-refinement-v1/val_reports/AF2CUE1_seed42_result.json'
PROJECT=resolve_drive_project_root(required_relative_paths=(ARCHIVE_REL,BASE_REL,SPDS_REL,CUE_REL))
ARCHIVE=require_project_artifact(PROJECT,ARCHIVE_REL)
DATA=WORK/'faruq-development-v3-grouped'
if not (DATA/'faruq_grouped_manifest.json').is_file():
    with tarfile.open(ARCHIVE,'r') as stream: stream.extractall(WORK,filter='data')
for required in (DATA/'data.yaml',DATA/'faruq_grouped_manifest.json',DATA/'val/images',DATA/'val/labels'):
    assert required.exists(), required
assert not (DATA/'test').exists(), 'STOP: test tidak boleh tersedia.'
ORIGINAL=PROJECT/'experiments/faruq-v3-af2-spds-v1'
REFINEMENT=PROJECT/'experiments/faruq-v3-af2-spds-refinement-v1'
OUTPUT=REFINEMENT/'val_reports'/'af2cue1_parent_bootstrap_validation.json'
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.analysis.af2cue1_parent_bootstrap import run_af2cue1_parent_bootstrap
result=run_af2cue1_parent_bootstrap(PROJECT,DATA,ORIGINAL,REFINEMENT,OUTPUT,device='0',iterations=1000,seed=20260829)
print('STATUS  :',result['exploratory_research_status'])
print('FORMAL  :',result['formal_frozen_decision'])
print('PARENTS :',result['paired_parent_bootstrap']['independent_parents'])
print('IMAGES  :',result['paired_parent_bootstrap']['validation_images'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_opened'])
print('SUMMARY :',OUTPUT)


In [ ]:
import pandas as pd
bootstrap=result['paired_parent_bootstrap']
rows=[]
for comparison,metrics in bootstrap['comparisons'].items():
    for metric,values in metrics.items(): rows.append({'comparison':comparison,'metric':metric,**values})
display(pd.DataFrame(rows).style.format({'point_delta':'{:+.2%}','ci95_low':'{:+.2%}','ci95_high':'{:+.2%}','probability_positive':'{:.2%}','probability_nonnegative':'{:.2%}'}))
print('CUSTOM POINT:',bootstrap['custom_point_metrics'])
print('REJECTED MISSING-CLASS SAMPLES:',bootstrap['rejected_missing_class_samples'])
print('Kirim tabel, status, jumlah parent/image, custom point, dan endpoint calibration. Jangan training atau membuka test.')
